In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import math
import warnings
import numpy
import pandas
import json
import re
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from torch.cuda.amp import autocast, GradScaler



In [2]:
warnings.filterwarnings("ignore", category = FutureWarning)
torch.set_float32_matmul_precision("high")


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")


Device in use: cuda


In [4]:
d_model = 384
d_ff = d_model * 4
num_heads = 6
num_layers = 4
memory_slots = 32
dropout = 0.1
max_seq_len = 512


In [5]:
def positional_encoder(seq_length, d_model):
    
    pe = torch.zeros(seq_length, d_model)
    pos = torch.arange(0, seq_length, dtype = torch.float32).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
    
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    
    return pe


In [ ]:
class DecoderNeuralMemory(nn.Module):
    
    """
    Slot-Addressed Differentiable Neural Working Memory.
    Maintains dynamic memory slots (memory_slots x d_model) per batch session.
    - Read: Queries historical memory slots via Multihead Cross-Attention
    - Write: Softmax-addressed gated associative update using incoming hidden states
    """
    
    def __init__(self, d_model = d_model, num_heads = num_heads, memory_slots = memory_slots, dropout = dropout):
        super(DecoderNeuralMemory, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.memory_slots = memory_slots
        
        # Learnable initial memory slots state
        
        self.init_memory = nn.Parameter(torch.randn(memory_slots, d_model) * 0.02)
        
        # Cross-Attention to read from memory
        
        self.read_attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        self.read_norm = nn.LayerNorm(d_model)
        
        # Projections for slot-addressed write
        
        self.write_key = nn.Linear(d_model, d_model, bias = False)
        self.write_val = nn.Linear(d_model, d_model, bias = False)
        self.write_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
            nn.Sigmoid()
        )
        
        self.mem_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def get_initial_state(self, batch_size, device):
        
        # [B, memory_slots, d_model]
        
        return self.init_memory.unsqueeze(0).expand(batch_size, -1, -1).clone().to(device)
        
    def read(self, x, memory_state):
        
        """
        x: [B, T, d_model]
        memory_state: [B, memory_slots, d_model]
        returns: context [B, T, d_model]
        """

        assert memory_state.size(-1) == self.d_model, f"Memory state mismatch: {memory_state.shape}"
        norm_x = self.read_norm(x)
        retrieved, _ = self.read_attn(norm_x, memory_state, memory_state)
        return self.dropout(retrieved)
        
    def write(self, hidden_states, memory_state, key_padding_mask = None):
        
        """
        hidden_states: [B, T, d_model] - current sequence representations
        memory_state: [B, memory_slots, d_model]
        key_padding_mask: [B, T] (bool mask where True = padding to ignore)
        """
        
        B, T, D = hidden_states.shape
        M = self.memory_slots
        
        # Keys and values from tokens to write
        
        k_write = self.write_key(hidden_states) # [B, T, D]
        v_write = self.write_val(hidden_states) # [B, T, D]
        
        # Softmax addressing matrix across memory slots: [B, M, T]
        # How relevant is token t to memory slot m?
        
        mem_keys = self.mem_norm(memory_state) # [B, M, D]
        scores = torch.bmm(mem_keys, k_write.transpose(1, 2)) / math.sqrt(D) # [B, M, T]
        
        if key_padding_mask is not None:
            
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1), float('-inf'))
            
        addr_weights = F.softmax(scores, dim = -1) # [B, M, T]
        # Replace NaN if all tokens were masked
        
        addr_weights = torch.nan_to_num(addr_weights, 0.0)
        
        # Candidate memory content per slot: [B, M, D]
        
        candidate = torch.bmm(addr_weights, v_write)
        
        # Slot-wise adaptive gating: [B, M, 1]
        
        gate_input = torch.cat([memory_state, candidate], dim = -1)
        alpha = self.write_gate(gate_input)
        
        # Gated memory update
        
        updated_memory = (1.0 - alpha) * memory_state + alpha * candidate
        updated_memory = self.mem_norm(updated_memory)
        assert updated_memory.size(-1) == self.d_model, f"Updated memory mismatch: {updated_memory.shape}"
        
        return updated_memory


In [7]:
class decoder_layer(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, dropout = dropout, num_heads = num_heads):
        super(decoder_layer, self).__init__()
        
        # layer norm
        
        self.norm = nn.LayerNorm(d_model)
        
        # Attn
        
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        
        # layer norm
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # FFN
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
        # dropout 
        
        self.dropout = nn.Dropout(dropout)
        
        # optimization
        
        self.apply(self.__init_weight__)
        
    def forward(self, x, attn_mask = None, key_padding_mask = None):
        
        # x -> norm
        
        norm = self.norm(x)
        
        # norm -> attn
        
        attn, _ = self.attn(
            norm,
            norm,
            norm,
            attn_mask = attn_mask,
            key_padding_mask = key_padding_mask
        )
        
        # residual
        
        x = x + self.dropout(attn)
        
        # residual -> norm
        
        norm2 = self.norm2(x)
        
        # norm -> FFN
        
        ffn = self.ffn(norm2)
        
        # residual
        
        x = x + self.dropout(ffn)
        
        return x
        
    
    def __init_weight__(self, modulue):
        
        for m in modulue.modules():
            
            if isinstance(m, nn.Linear):
                
                nn.init.normal_(m.weight, 0, 0.02)
                
                if m.bias is not None:
                    
                    nn.init.zeros_(m.bias)
                    
            if isinstance(m, nn.MultiheadAttention):
                
                nn.init.normal_(m.in_proj_weight, 0, 0.02)
                
                if m.in_proj_bias is not None:
                    
                    nn.init.zeros_(m.in_proj_bias)
                
                nn.init.normal_(m.out_proj.weight, 0, 0.02)
                
                if m.out_proj.bias is not None:
                    
                    nn.init.zeros_(m.out_proj.bias)


In [8]:
class decoder(nn.Module):
    
    def __init__(self, vocab_size, d_model = d_model, d_ff = d_ff, num_layers = num_layers, num_heads = num_heads, memory_slots = memory_slots, dropout = dropout):
        super(decoder, self).__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Embedding layer
        self.embed = nn.Embedding(vocab_size, d_model)
        
        # Neural Working Memory System (persists and tracks context across chunks/turns)
        self.memory = DecoderNeuralMemory(d_model = d_model, num_heads = num_heads, memory_slots = memory_slots, dropout = dropout)
        
        # Transformer Decoder Layers
        self.layers = nn.ModuleList([
            decoder_layer(d_model = d_model, d_ff = d_ff, num_heads = num_heads, dropout = dropout) for _ in range(num_layers)
        ])
        
        # Final LayerNorm
        self.norm = nn.LayerNorm(d_model)
        
        # Output projection head (tied to embedding weights)
        self.head = nn.Linear(d_model, vocab_size, bias = False)
        self.head.weight = self.embed.weight
        
    def forward(self, x, attn_mask = None, key_padding_mask = None, memory_state = None, update_memory = True, inspect = False):
        
        B, seq_length = x.shape
        
        # Initialize memory state if not provided
        if memory_state is None:
            memory_state = self.memory.get_initial_state(B, x.device)
            
        # Embedding + Positional Encoding
        embed = self.embed(x) * math.sqrt(self.d_model)
        pos = positional_encoder(seq_length, self.d_model).to(x.device)
        h = embed + pos
        
        # Read relevant historical context from Neural Working Memory
        mem_context = self.memory.read(h, memory_state)
        h = h + mem_context
        
        if inspect: print(f'h size: {h.size()} | h shape: {h.shape}')
        
        # Standard Causal Attention Mask for local window
        if attn_mask is None and seq_length > 1:
            attn_mask = torch.triu(torch.ones((seq_length, seq_length), device = x.device), diagonal = 1).bool()
        
        # Pass through Decoder layers
        for layer in self.layers:
            h = layer(
                h,
                attn_mask = attn_mask,
                key_padding_mask = key_padding_mask
            )
            
        # Update Neural Memory with current chunk's contextual representations
        if update_memory:
            memory_state = self.memory.write(h, memory_state, key_padding_mask = key_padding_mask)
            
        # Output projection
        h = self.norm(h)
        logits = self.head(h)
            
        return logits, memory_state


In [9]:
# ── Custom 16K BPE Tokenizer ────────────────────────────────────────────────
# Trains from dataset text → saves to cherry_tokenizer/
# Idempotent: if cherry_tokenizer/ already exists, loads directly.

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

TOKENIZER_PATH = Path(r"/kaggle/input/datasets/binnyshukla/tokeinzer")
DATASET_DIR    = Path(r"/kaggle/input/datasets/binnyshukla/cherry-agentic-dataset")
VOCAB_SIZE     = 16_000

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|im_start|>",
    "<|im_end|>",
    "<|analysis|>",
    "<|analysis_end|>",
    "<|tool_call|>",
    "<|tool_end|>",
]

def _iter_texts(dataset_dir):
    for split in ["train.jsonl", "validation.jsonl"]:
        p = dataset_dir / split
        if not p.exists():
            continue
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    for turn in record.get("turns", []):
                        yield turn.get("content", "")

def train_custom_tokenizer(dataset_dir, tokenizer_path, vocab_size):
    if tokenizer_path.exists() and (tokenizer_path / "tokenizer.json").exists():
        print(f"[tokenizer] Loading existing tokenizer from {tokenizer_path}")
        return PreTrainedTokenizerFast.from_pretrained(str(tokenizer_path))

    print(f"[tokenizer] Training {vocab_size:,}-token BPE tokenizer from {dataset_dir}...")
    bpe = Tokenizer(models.BPE(unk_token="<|unk|>"))
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.decoder       = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size     = vocab_size,
        special_tokens = SPECIAL_TOKENS,
        min_frequency  = 2,
        show_progress  = True,
    )

    bpe.train_from_iterator(_iter_texts(dataset_dir), trainer=trainer)

    hf_tok = PreTrainedTokenizerFast(
        tokenizer_object = bpe,
        bos_token        = "<|im_start|>",
        eos_token        = "<|im_end|>",
        pad_token        = "<|pad|>",
        unk_token        = "<|unk|>",
    )

    tokenizer_path.mkdir(parents=True, exist_ok=True)
    hf_tok.save_pretrained(str(tokenizer_path))
    print(f"[tokenizer] Saved to {tokenizer_path}")
    return hf_tok


In [10]:
tokenizer = train_custom_tokenizer(DATASET_DIR, TOKENIZER_PATH, VOCAB_SIZE)

pad_id       = tokenizer.convert_tokens_to_ids("<|pad|>")
im_end_id    = tokenizer.convert_tokens_to_ids("<|im_end|>")
tool_call_id = tokenizer.convert_tokens_to_ids("<|tool_call|>")
tool_end_id  = tokenizer.convert_tokens_to_ids("<|tool_end|>")

vocab_size = len(tokenizer)

print(f"Vocabulary size: {vocab_size:,}")
print(f"pad_id={pad_id}  im_end_id={im_end_id}  tool_call_id={tool_call_id}  tool_end_id={tool_end_id}")


[tokenizer] Loading existing tokenizer from /kaggle/input/datasets/binnyshukla/tokeinzer
Vocabulary size: 16,000
pad_id=0  im_end_id=3  tool_call_id=6  tool_end_id=7


In [11]:
Decoder = decoder(
    vocab_size = vocab_size,
    d_model = d_model,
    d_ff = d_ff,
    num_layers = num_layers,
    num_heads = num_heads,
    memory_slots = memory_slots,
    dropout = dropout
)

Decoder = nn.DataParallel(Decoder, device_ids=[0, 1])

Decoder = Decoder.to("cuda")

print(Decoder)
print("--------------------------------------------------------------------")
params = sum(p.numel() for p in Decoder.parameters())
print(f"No. of params in Decoder (d_model={d_model}, num_heads={num_heads}): {params:,}")


DataParallel(
  (module): decoder(
    (embed): Embedding(16000, 384)
    (memory): DecoderNeuralMemory(
      (read_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (read_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (write_key): Linear(in_features=384, out_features=384, bias=False)
      (write_val): Linear(in_features=384, out_features=384, bias=False)
      (write_gate): Sequential(
        (0): Linear(in_features=768, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=384, out_features=1, bias=True)
        (3): Sigmoid()
      )
      (mem_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-3): 4 x decoder_layer(
        (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (

In [12]:
# Params

learning_rate = 3e-4

warmup_epoch = 3
plateau_epochs = 4
decay_epoch = 3
Epoch = 10

# Loss

model_loss = nn.CrossEntropyLoss(ignore_index = -100)

# optimizers

optimizer = optim.AdamW(Decoder.parameters(), lr = learning_rate)

# scheduler

early_scheduler = optim.lr_scheduler.LinearLR(optimizer, total_iters = warmup_epoch)
plateau_scheduler = optim.lr_scheduler.ConstantLR(
    optimizer,
    factor=1.0,
    total_iters=plateau_epochs
)
later_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, decay_epoch, eta_min = 1e-5)
scheduler = optim.lr_scheduler.SequentialLR(optimizer, [early_scheduler, plateau_scheduler, later_scheduler], [warmup_epoch, warmup_epoch + plateau_epochs])


In [ ]:
import re
from torch.utils.data import Dataset, DataLoader

allowed_roles = {'user', 'assistant', 'tool'}

def clean_assistant_content(content):
    # Removes intermediate reasoning tags from loss if desired, or keeps them clean
    return re.sub(
        r'<\|analysis\|>.*?<\|analysis_end\|>\s*',
        '',
        content,
        flags = re.DOTALL
    )

class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_seq_len = 512, chunk_stride = 256):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.chunk_stride = chunk_stride
        self.samples = []
        
        print(f"Loading and tokenizing {Path(jsonl_path).name} (max_seq_len={max_seq_len})...")
        
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                if "turns" not in record:
                    continue
                
                chunks = self.encode_and_chunk(record["turns"])
                for input_ids, labels in chunks:
                    self.samples.append({
                        "input_ids": input_ids,
                        "labels": labels
                    })
                    
        print(f"  -> Total training sequences: {len(self.samples):,}")

    def encode_and_chunk(self, turns):
        full_input_ids = []
        full_labels = []

        for turn in turns:
            role = turn["role"]
            prefix = f"<|im_start|>{role}\n"
            content = clean_assistant_content(turn["content"]) if role == "assistant" else turn["content"]
            suffix = f"{content}\n<|im_end|>\n"

            prefix_ids = self.tokenizer.encode(prefix, add_special_tokens=False)
            suffix_ids = self.tokenizer.encode(suffix, add_special_tokens=False)

            full_input_ids.extend(prefix_ids)
            full_input_ids.extend(suffix_ids)

            if role == "assistant":
                full_labels.extend([-100] * len(prefix_ids))
                full_labels.extend(suffix_ids)
            else:
                full_labels.extend([-100] * (len(prefix_ids) + len(suffix_ids)))

        if len(full_input_ids) < 2:
            return []

        # If fits inside max_seq_len, return directly
        if len(full_input_ids) <= self.max_seq_len:
            return [(full_input_ids, full_labels)]

        # Sliding window chunking for long trajectories
        chunks = []
        for start in range(0, len(full_input_ids), self.chunk_stride):
            end = start + self.max_seq_len
            c_inputs = full_input_ids[start:end]
            c_labels = full_labels[start:end]
            if len(c_inputs) >= 32:  # Avoid tiny trailing fragments
                chunks.append((c_inputs, c_labels))
            if end >= len(full_input_ids):
                break
        return chunks

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    padded_inputs = []
    padded_targets = []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        padded_inputs.append(item["input_ids"] + [pad_id] * pad_len)
        padded_targets.append(item["labels"] + [-100] * pad_len)

    return torch.tensor(padded_inputs, dtype=torch.long), torch.tensor(padded_targets, dtype=torch.long)

# ── Initialize DataLoaders with Unified Dataset ─────────────────────────────

dataset_dir = Path(r"/kaggle/input/datasets/binnyshukla/cherry-agentic-dataset")

train_set = SFTDataset(dataset_dir / "train.jsonl", tokenizer, max_seq_len = 2048)
val_set   = SFTDataset(dataset_dir / "validation.jsonl", tokenizer, max_seq_len = 2048)

batch_size = 8
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"DataLoaders Ready: {len(train_set):,} Train sequences | {len(val_set):,} Validation sequences")


Loading and tokenizing train.jsonl (max_seq_len=2048)...
  -> Total training sequences: 59,571
Loading and tokenizing validation.jsonl (max_seq_len=2048)...
  -> Total training sequences: 6,896
DataLoaders Ready: 59,571 Train sequences | 6,896 Validation sequences


In [14]:
vocab_output = 16000


In [ ]:
cpu_device = torch.device("cpu")
print(f"Device: {cpu_device}")


Device: cpu


In [ ]:

print("Starting Decoder training...")

best_val_loss = float("inf")
scaler = GradScaler()   # AMP gradient scaler

for epoch in range(1, Epoch + 1):
    
    Decoder.train()
    total_train_loss = 0.0
    
    for inputs, targets in train_loader:
        
        inputs = inputs.to("cuda")
        targets = targets.to("cuda")
        
        optimizer.zero_grad()
        
        key_padding_mask = inputs.eq(pad_id)
        
        # forward pass under autocast
        with autocast():
            logits, memory_state = Decoder(
                inputs,
                key_padding_mask=key_padding_mask,
                memory_state=None,
                update_memory=True
            )
            
            # Shift logits and targets for next-token prediction
            shift_logits = logits[:, :-1, :].contiguous()
            shift_targets = targets[:, 1:].contiguous()
            
            loss = model_loss(
            shift_logits.view(-1, logits.size(-1)),
            shift_targets.view(-1),
        )
        
        # backward pass with scaled gradients
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(Decoder.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        
    
    scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    Decoder.eval()
    total_val_loss = 0.0
    

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to("cuda")
            targets = targets.to("cuda")
            key_padding_mask = inputs.eq(pad_id)
    
            with autocast():
                logits, _ = Decoder(
                    inputs,
                    key_padding_mask=key_padding_mask,
                    memory_state=None,  # important: reset every batch
                    update_memory=True,
                )
    
                shift_logits = logits[:, :-1, :].contiguous()
                shift_targets = targets[:, 1:].contiguous()
    
                loss = model_loss(
                    shift_logits.view(-1, logits.size(-1)),
                    shift_targets.view(-1),
                )
    
            total_val_loss += loss.item()
            

    
    avg_val_loss = total_val_loss / len(val_loader)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(Decoder.state_dict(), "best_decoder_mem.pt")
        checkpoint_status = "SAVED"
    else:
        checkpoint_status = ""
    
    print(
        f"Epoch {epoch:02d}/{Epoch:02d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Best Val: {best_val_loss:.4f} "
        f"[{checkpoint_status}]"
    )


Starting Decoder training...
Epoch 01/10 | Train Loss: 45.2153 | Val Loss: 7.0655 | Best Val: 7.0655 [SAVED]
Epoch 02/10 | Train Loss: 6.6646 | Val Loss: 6.0078 | Best Val: 6.0078 [SAVED]
Epoch 03/10 | Train Loss: 6.0049 | Val Loss: 5.5684 | Best Val: 5.5684 [SAVED]
Epoch 04/10 | Train Loss: 5.4727 | Val Loss: 5.1915 | Best Val: 5.1915 [SAVED]
Epoch 05/10 | Train Loss: 5.0585 | Val Loss: 4.8026 | Best Val: 4.8026 [SAVED]
Epoch 06/10 | Train Loss: 4.7492 | Val Loss: 4.5582 | Best Val: 4.5582 [SAVED]
Epoch 07/10 | Train Loss: 4.4735 | Val Loss: 4.3025 | Best Val: 4.3025 [SAVED]
Epoch 08/10 | Train Loss: 4.2980 | Val Loss: 4.2284 | Best Val: 4.2284 [SAVED]
Epoch 09/10 | Train Loss: 4.2232 | Val Loss: 4.1674 | Best Val: 4.1674 [SAVED]
Epoch 10/10 | Train Loss: 4.1785 | Val Loss: 4.1478 | Best Val: 4.1478 [SAVED]


In [ ]:
import os, shutil

src = "/kaggle/working/best_decoder_128.pt"
dst_dir = "/kaggle/outputs"
os.makedirs(dst_dir, exist_ok=True)

if os.path.exists(src):
    shutil.copy(src, os.path.join(dst_dir, "best_decoder_128.pt"))
    print("Model copied successfully!")
else:
    print("Model file not found at", src)


Model file not found at /kaggle/working/best_decoder_128.pt
